# Unidad 3 – Sesión 2
## Transformación de variables para análisis y modelado

**Curso:** Minería de Datos  
**Fase CRISP-DM:** Preparación de los Datos

### Propósito
- corregir tipos de datos;
- transformar fechas;
- crear variables derivadas;
- codificar variables categóricas;
- escalar variables numéricas;
- validar las transformaciones.

> **Idea clave:** Limpiar los datos no es suficiente; también debemos representarlos de una forma útil para el análisis.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "cliente_id": [101, 102, 103, 104, 105],
    "edad": ["25", "32", "40", "29", "36"],
    "ciudad": ["LIMA", "CUSCO", "LIMA", "AREQUIPA", "CUSCO"],
    "genero": ["M", "F", "F", "M", "F"],
    "fecha_registro": ["2026-01-10","2026-02-15","2026-02-20","2026-03-05","2026-03-21"],
    "ingreso": [1500, 3200, 2100, 1800, 4500],
    "activo": ["SI", "NO", "SI", "SI", "NO"]
}

df = pd.DataFrame(data)
df

# 1. Revisar tipos de datos

In [ ]:
df.dtypes

`edad` es conceptualmente numérica aunque Pandas la detecte como texto.  
`fecha_registro` debería ser temporal.  
`cliente_id` es numérico técnicamente, pero funciona como identificador.

# 2. Convertir texto a numérico

In [ ]:
df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
df.dtypes

In [ ]:
ejemplo = pd.Series(["25", "32", "ERROR", "40"])
pd.to_numeric(ejemplo, errors="coerce")

`errors="coerce"` convierte valores no válidos a `NaN` en lugar de detener la ejecución.

# 3. Convertir fechas

In [ ]:
df["fecha_registro"] = pd.to_datetime(df["fecha_registro"], errors="coerce")
df.dtypes

# 4. Extraer información temporal

In [ ]:
df["anio_registro"] = df["fecha_registro"].dt.year
df["mes_registro"] = df["fecha_registro"].dt.month
df["dia_registro"] = df["fecha_registro"].dt.day
df["dia_semana"] = df["fecha_registro"].dt.day_name()
df["periodo"] = (
    df["anio_registro"].astype(str)
    + "-"
    + df["mes_registro"].astype(str).str.zfill(2)
)
df[["fecha_registro","anio_registro","mes_registro","dia_registro","dia_semana", "periodo"]]


### Pregunta
¿Por qué `mes_registro` o `dia_semana` pueden ser más útiles que conservar solamente la fecha completa?

# 5. Codificación binaria

In [ ]:
df["activo_num"] = df["activo"].map({"SI": 1, "NO": 0})
df[["activo", "activo_num"]]

In [ ]:
df["genero_num"] = df["genero"].map({"M": 0, "F": 1})
df[["genero", "genero_num"]]

Los valores 0 y 1 son códigos. No implican una jerarquía entre categorías.

# 6. One-Hot Encoding

In [ ]:
ciudades = pd.get_dummies(df["ciudad"], prefix="ciudad", dtype=int)
ciudades

In [ ]:
df = pd.concat([df, ciudades], axis=1)
df.head()

Para variables nominales como ciudad, One-Hot Encoding evita crear un orden artificial.

# 7. Variables ordinales

In [ ]:
riesgo = pd.Series(["BAJO", "MEDIO", "ALTO", "MEDIO", "BAJO"])

riesgo_num = riesgo.map({
    "BAJO": 1,
    "MEDIO": 2,
    "ALTO": 3
})

pd.DataFrame({"riesgo": riesgo, "riesgo_num": riesgo_num})

Aquí el orden sí tiene sentido: `BAJO < MEDIO < ALTO`.

# 8. Error frecuente: códigos para categorías nominales

In [ ]:
df["codigo_ciudad"] = df["ciudad"].astype("category").cat.codes
df[["ciudad","codigo_ciudad"]]

Aunque técnicamente funciona, puede inducir un orden que no existe. Para variables nominales es preferible One-Hot Encoding.

# 9. Escalamiento numérico

In [ ]:
df[["edad","ingreso"]].describe()

# 10. Normalización Min-Max

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
df[["edad_norm","ingreso_norm"]] = minmax.fit_transform(df[["edad","ingreso"]])

df[["edad","edad_norm","ingreso","ingreso_norm"]]

Min-Max lleva normalmente los valores al rango de 0 a 1.

# 11. Estandarización

In [ ]:
from sklearn.preprocessing import StandardScaler

standard = StandardScaler()
df[["edad_std","ingreso_std"]] = standard.fit_transform(df[["edad","ingreso"]])

df[["edad","edad_std","ingreso","ingreso_std"]]

La estandarización deja las variables aproximadamente con media 0 y desviación estándar 1.

# 12. Normalización vs estandarización

In [ ]:
pd.DataFrame({
    "Tecnica": ["Min-Max","StandardScaler","Sin escalamiento"],
    "Resultado": [
        "Valores normalmente entre 0 y 1",
        "Media ≈ 0 y desviación estándar ≈ 1",
        "Valores originales"
    ],
    "Uso_general": [
        "Cuando se requiere una escala definida",
        "Algoritmos sensibles a escala",
        "Modelos donde la escala no es crítica"
    ]
})

No todos los modelos necesitan escalamiento. Los árboles de decisión, por ejemplo, no dependen de la escala de la misma forma que KNN o K-Means.

# 13. No transformar identificadores

In [ ]:
df[["cliente_id"]]

`cliente_id` no debe normalizarse ni interpretarse como una cantidad solo porque sea numérico.

# 14. Construcción del dataset preparado

In [ ]:
columnas_modelo = [
    "edad",
    "ingreso",
    "activo_num",
    "ciudad_AREQUIPA",
    "ciudad_CUSCO",
    "ciudad_LIMA"
]

df_preparado = df[columnas_modelo].copy()
df_preparado

# 15. Validación

In [ ]:
df_preparado.dtypes

In [ ]:
df_preparado.isnull().sum()

In [ ]:
df_preparado.describe()

Después de transformar debemos verificar:
- tipos;
- nulos inesperados;
- codificación;
- escalas;
- columnas generadas.

# 16. Actividad práctica con dataset propio

In [ ]:
import pandas as pd
import numpy as np

ruta = "./LI-Small_Trans.csv"
df_proyecto = pd.read_csv(ruta)
print(df_proyecto.shape)
df_proyecto.head()

In [ ]:
df_proyecto = df_proyecto.rename(columns={
    "From Bank": "banco_origen",
    "Account": "cuenta_origen",
    "To Bank": "banco_destino",
    "Account.1": "cuenta_destino",
})
df_proyecto.dtypes

Identifica como mínimo:
- 1 variable que requiera conversión de tipo;
- 1 variable temporal;
- 2 categóricas;
- 2 numéricas;
- 1 identificador.

In [ ]:
filas = [
    ["Timestamp", "object", "datetime64", "pd.to_datetime(format='%Y/%m/%d %H:%M')", "fecha de la transacción, permite análisis temporal"],
    ["banco_origen / banco_destino", "int64", "object", "astype(str)", "código de banco: identificador, no cantidad (demasiados valores para One-Hot)"],
    ["cuenta_origen / cuenta_destino", "object", "object", "sin transformación", "identificador de cuenta, no se normaliza"],
    ["Payment Format", "object", "columnas 0/1", "get_dummies (One-Hot)", "categórica nominal sin orden"],
    ["Receiving Currency", "object", "columnas 0/1", "get_dummies (One-Hot)", "categórica nominal sin orden (15 monedas)"],
    ["Receiving Currency vs Payment Currency", "object", "binaria 0/1", "comparación ==", "indica si hubo cambio de moneda"],
    ["Amount Received / Amount Paid", "float64", "estandarizada", "log1p + StandardScaler", "montos muy asimétricos y en escalas grandes"],
    ["Is Laundering", "int64", "sin cambio", "ninguna", "variable objetivo"],
]

plan_transformacion = pd.DataFrame(filas, columns=["Variable", "Tipo_actual", "Tipo_esperado", "Transformacion", "Justificacion"])
plan_transformacion

Aplica como mínimo 4 transformaciones:
1. conversión de tipo;
2. transformación temporal;
3. codificación categórica;
4. escalamiento numérico.

In [ ]:
# Convierte la columna Timestamp a formato datetime
df_proyecto["Timestamp"] = pd.to_datetime(
    df_proyecto["Timestamp"], errors="coerce"
)

# Verifica los tipos de datos actualizados
df_proyecto.dtypes

In [ ]:
import pandas as pd

# 1. Asegúrate de que la columna de fecha esté en formato datetime dentro de df_proyecto
df_proyecto["Timestamp"] = pd.to_datetime(df_proyecto["Timestamp"])

# 2. Extrae la información temporal usando df_proyecto
df_proyecto["anio_registro"] = df_proyecto["Timestamp"].dt.year
df_proyecto["mes_registro"] = df_proyecto["Timestamp"].dt.month
df_proyecto["dia_registro"] = df_proyecto["Timestamp"].dt.day
df_proyecto["dia_semana"] = df_proyecto["Timestamp"].dt.day_name()

# 3. Verifica el resultado con las columnas que necesites
df_proyecto[
    [
        "Timestamp",
        "anio_registro",
        "mes_registro",
        "dia_registro",
        "dia_semana",
    ]
].head()

In [ ]:
# Cuenta cuántas veces aparece cada valor único en la columna Is Laundering
df_proyecto["Is Laundering"].value_counts()

In [ ]:
# Muestra el resumen estadístico de las columnas de montos
df_proyecto[["Amount Received", "Amount Paid"]].describe()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Crear la instancia del MinMaxScaler
minmax = MinMaxScaler()

# 2. Aplicar la transformación y guardar en nuevas columnas
df_proyecto[["Amount_Received_norm", "Amount_Paid_norm"]] = (
    minmax.fit_transform(df_proyecto[["Amount Received", "Amount Paid"]])
)

# 3. Visualizar el resultado comparando las columnas originales con las normalizadas
df_proyecto[
    [
        "Amount Received",
        "Amount_Received_norm",
        "Amount Paid",
        "Amount_Paid_norm",
    ]
].head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Configurar Pandas para mostrar decimales normales en lugar de notación científica
pd.options.display.float_format = "{:.6f}".format

# 2. Crear la instancia del MinMaxScaler
minmax = MinMaxScaler()

# 3. Aplicar la transformación y guardar en las nuevas columnas
df_proyecto[["Amount_Received_norm", "Amount_Paid_norm"]] = (
    minmax.fit_transform(df_proyecto[["Amount Received", "Amount Paid"]])
)

# 4. Visualizar el resultado igual que en el ejemplo
df_proyecto[
    [
        "Amount Received",
        "Amount_Received_norm",
        "Amount Paid",
        "Amount_Paid_norm",
    ]
].head()

# 17. Matriz final de transformación

In [ ]:
filas = [
    ["Timestamp", "object (texto)", "pd.to_datetime + hora y dia_semana", "datetime64 + hora (0-23)", "permite análisis por hora y día"],
    ["banco_origen / banco_destino", "int64", "astype(str)", "object (identificador)", "código, no cantidad; demasiados valores para One-Hot"],
    ["cuenta_origen / cuenta_destino", "object", "ninguna", "object (identificador)", "no se normaliza ni codifica"],
    ["Payment Format", "object (7 valores)", "One-Hot (get_dummies)", "7 columnas 0/1 formato_*", "variable nominal sin orden"],
    ["Receiving Currency", "object (15 valores)", "One-Hot (get_dummies)", "15 columnas 0/1 moneda_*", "variable nominal sin orden"],
    ["Receiving Currency vs Payment Currency", "object", "comparación", "misma_moneda 0/1", "indica cambio de moneda"],
    ["Amount Received / Amount Paid", "float64 (escala muy amplia)", "log1p + StandardScaler", "monto_recibido_std / monto_pagado_std", "reduce asimetría y diferencia de escala"],
    ["Is Laundering", "int64", "ninguna", "int64 (0/1)", "variable objetivo; muy desbalanceada (~0.05% positivos)"],
]

matriz_transformacion = pd.DataFrame(filas, columns=["Variable", "Estado_original", "Transformacion_aplicada", "Estado_final", "Justificacion"])
matriz_transformacion

In [ ]:
# 1. Conversión de tipo: códigos de banco son identificadores
df_proyecto[["banco_origen", "banco_destino"]] = df_proyecto[["banco_origen", "banco_destino"]].astype(str)

# 2. Transformación temporal
df_proyecto["fecha"] = pd.to_datetime(df_proyecto["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")
df_proyecto["hora"] = df_proyecto["fecha"].dt.hour
df_proyecto["dia_semana"] = df_proyecto["fecha"].dt.day_name()

# 3. Codificación categórica
df_proyecto["misma_moneda"] = (df_proyecto["Receiving Currency"] == df_proyecto["Payment Currency"]).astype(int)
formatos = pd.get_dummies(df_proyecto["Payment Format"], prefix="formato", dtype=int)
monedas = pd.get_dummies(df_proyecto["Receiving Currency"], prefix="moneda", dtype=int)
df_proyecto = pd.concat([df_proyecto, formatos, monedas], axis=1)

# 4. Escalamiento numérico (log1p por asimetría de montos, luego StandardScaler)
from sklearn.preprocessing import StandardScaler

montos = ["Amount Received", "Amount Paid"]
df_proyecto[["monto_recibido_log", "monto_pagado_log"]] = np.log1p(df_proyecto[montos])
escala = StandardScaler()
df_proyecto[["monto_recibido_std", "monto_pagado_std"]] = escala.fit_transform(
    df_proyecto[["monto_recibido_log", "monto_pagado_log"]]
)

# Dataset preparado y validación
df_preparado_proyecto = df_proyecto[
    ["hora", "misma_moneda", "monto_recibido_std", "monto_pagado_std"]
    + list(formatos.columns) + list(monedas.columns) + ["Is Laundering"]
].copy()
print(df_preparado_proyecto.dtypes)
print(df_preparado_proyecto.isnull().sum())
df_preparado_proyecto.describe()

### Ejemplo de estructura

| Variable | Estado original | Transformación | Estado final | Justificación |
|---|---|---|---|---|
| edad | object | to_numeric() | numérica | representa años |
| fecha | object | to_datetime() | datetime | análisis temporal |
| ciudad | texto | One-Hot | columnas 0/1 | variable nominal |
| activo | SI/NO | map | 0/1 | variable binaria |
| ingreso | escala original | StandardScaler | estandarizada | evitar diferencia de escala |

# 18. Preguntas de cierre

1. ¿Por qué un identificador no debería normalizarse?
   Porque es una clave primaria
3. ¿Cuándo utilizar One-Hot Encoding?
   Transformar variables categóricas nominales en un formato numérico que los algoritmos de Machine Learning puedan interpretar correctamente.
5. ¿Qué problema existe al codificar Lima=1, Cusco=2 y Arequipa=3?
   Puede introducir un orden que no existe
7. ¿Qué diferencia existe entre una variable nominal y una ordinal?
   La ordinal tiene un orden o jerarquia mientras que la nominal no, pese a ambas ser categoricas
9. ¿Qué información puede extraerse de una fecha?
   Componentes del calendario, día de la semana, indicadores como fin de semana o feriado, y tiempo transcurrido.
11. ¿Normalizar siempre mejora un dataset?
    No, en modelos basados en árboles o cuando las características ya comparten la misma escala no aporta beneficios.
13. ¿Cuál es la diferencia entre normalización y estandarización?
    La normalización escala los valores a un rango fijo como 0 a 1, la estandarización ajusta los datos a media 0 y desviación estándar 1.
15. ¿Por qué debemos conservar el dataset original?
    Para asegurar la reproducibilidad, auditoría, corregir errores en las transformaciones y probar nuevos enfoques sin perder la fuente.
17. ¿Qué riesgo existe al transformar variables sin documentar?
    Perder la trazabilidad, causar fuga de datos, dificultar la interpretación y fallar al desplegar el modelo en producción.
19. ¿Qué validaciones deben realizarse después de transformar?
    Revisar ausencia de nulos y duplicados, validar las dimensiones del dataset y comprobar que los rangos o escalas sean los esperados.

# 19. Producto final

Cada grupo debe entregar:
- dataset original;
- dataset preparado;
- plan de transformación;
- matriz final de transformación;
- conclusión de máximo 5 líneas.

> **Conclusión:** ¿Qué transformaciones fueron necesarias y por qué mejoran la utilidad del dataset?
> Al transformar las fechas en datos temporales, pudimos ordenar la información y extraer con facilidad detalles clave como el año, mes y día. Además, al normalizar los montos con el escalamiento y mantener intactos los códigos de identificación, logramos dejar el dataset totalmente preparado y listo para cualquier análisis futuro.

# 20. Cierre

### 20. Cierre – Resultados obtenidos

En esta sesión trabajamos con un dataset de **6 924 049 transacciones financieras** y 11 columnas originales.

1. Convertimos las fechas y extrajimos información temporal, como la hora y el día de la semana.
2. Aplicamos One-Hot Encoding a 7 formatos de pago y 15 tipos de moneda, además de estandarizar los montos.
3. Obtuvimos un dataset preparado con **27 columnas y sin valores nulos**.
4. Encontramos **3 565 transacciones etiquetadas como lavado de dinero**, equivalentes aproximadamente al 0,0515 % del total.
5. Concluimos que las transformaciones facilitaron el análisis de los datos, aunque existe un fuerte desbalance entre las transacciones normales y las de lavado de dinero.